# 08 · Artifacts, the DAG & state

Every dbt command writes JSON **artifacts** into `target/`:

| artifact | written by | contains |
|---|---|---|
| `manifest.json` | every command | every node, config, and DAG edge |
| `run_results.json` | run/build/test/... | per-node status + timing |
| `catalog.json` | `docs generate` | column types & stats from the warehouse |
| `sources.json` | `source freshness` | freshness results |

Everything dbt knows is in these files -- the docs site, `state:modified`,
and half the dbt ecosystem are built on them.

Companion reading: [docs/17](../docs/17_debugging_and_performance.md) and
[docs/16](../docs/16_state_defer_ci.md).

In [ ]:
# --- Standard setup (identical in every notebook) ---------------------------
import os, sys, json, subprocess
from pathlib import Path
from datetime import date, timedelta

from dotenv import load_dotenv

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
load_dotenv(REPO / ".env")

os.chdir(REPO / "jaffle_shop")          # dbt must run from the project dir
os.environ.setdefault("DBT_PROFILES_DIR", str(REPO / "jaffle_shop"))

from dbt.cli.main import dbtRunner

def run_dbt(args):
    """Run dbt programmatically (same engine as the CLI). Returns a
    dbtRunnerResult: .success says how it went, .result has per-node details.
    Crucially it never sys.exit()s -- perfect for notebooks."""
    print(f"$ dbt {' '.join(args)}")
    res = dbtRunner().invoke(args)
    print(f"-> success={res.success}")
    return res

def load_day(*flags):
    """Invoke the raw-data generator (plays the role of the EL tool)."""
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "generate_data.py"), *flags],
        check=True,
    )


In [ ]:
# --- Warehouse connection for %%sql cells (jupysql) and pandas --------------
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBT_USER', 'dbt')}:{os.getenv('DBT_PASSWORD', 'dbt')}"
    f"@{os.getenv('DBT_HOST', 'localhost')}:{os.getenv('DBT_PORT', '5432')}"
    f"/{os.getenv('DBT_DBNAME', 'jaffle_shop')}"
)

%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 25


## 1. Refresh all artifacts

In [ ]:
res = run_dbt(["build"])
assert res.success
res = run_dbt(["docs", "generate"])
assert res.success

## 2. Parse the manifest and draw the DAG (no graphviz needed)

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

manifest = json.loads(Path("target/manifest.json").read_text())

LAYER = {"source": 0, "seed": 0, "staging": 1, "intermediate": 2,
         "marts": 3, "snapshot": 1, "exposure": 4}
PALETTE = {0: "#8ecae6", 1: "#219ebc", 2: "#ffb703", 3: "#fb8500", 4: "#c1121f"}

def layer_of(uid, node):
    rt = node["resource_type"]
    if rt in ("source", "seed", "snapshot", "exposure"):
        return LAYER[rt]
    return LAYER.get(node["fqn"][1] if len(node["fqn"]) > 2 else "marts", 3)

G = nx.DiGraph()
pool = {**manifest["nodes"], **manifest["sources"], **manifest["exposures"]}
for uid, node in pool.items():
    if node["resource_type"] in ("model", "seed", "snapshot", "source", "exposure"):
        G.add_node(uid, label=node["name"], layer=layer_of(uid, node))
for child, parents in {**manifest["parent_map"]}.items():
    for parent in parents:
        if child in G and parent in G:
            G.add_edge(parent, child)

pos = nx.multipartite_layout(G, subset_key="layer")
plt.figure(figsize=(14, 9))
colors = [PALETTE[G.nodes[n]["layer"]] for n in G.nodes]
nx.draw_networkx(G, pos, node_color=colors, node_size=1800, font_size=7,
                 labels={n: G.nodes[n]["label"] for n in G.nodes},
                 edge_color="#adb5bd", arrowsize=12)
plt.title("jaffle_shop DAG (from manifest.json)")
plt.axis("off")
plt.tight_layout()

Left to right: sources/seeds -> staging -> intermediate -> marts -> the
exposure (this very notebook family). `ref()`/`source()` calls are the only
thing that created those edges.

## 3. Where does build time go? (`run_results.json`)

In [ ]:
rr = json.loads(Path("target/run_results.json").read_text())
timings = (pd.DataFrame(
    [{"node": r["unique_id"].split(".")[-1], "seconds": r["execution_time"],
      "status": r["status"]} for r in rr["results"]])
    .sort_values("seconds", ascending=False).head(15))

timings.plot.barh(x="node", y="seconds", legend=False, figsize=(8, 5),
                  color="#219ebc", title="slowest nodes, last build")
plt.gca().invert_yaxis()
plt.tight_layout()

## 4. Warehouse stats from `catalog.json`

In [ ]:
catalog = json.loads(Path("target/catalog.json").read_text())
rows = []
for uid, node in catalog["nodes"].items():
    stats = node.get("stats", {})
    rows.append({"relation": node["metadata"]["schema"] + "." + node["metadata"]["name"],
                 "type": node["metadata"]["type"],
                 "columns": len(node["columns"])})
pd.DataFrame(rows).sort_values("relation").reset_index(drop=True)

## 5. State comparison: the trick behind slim CI

`state:modified` compares the CURRENT project against a SAVED manifest and
selects only what changed. Save state, "edit" a model, ask dbt what's
affected:

In [ ]:
import shutil

shutil.copytree("target", "target_base", dirs_exist_ok=True)   # save state

model = Path("models/marts/dim_products.sql")
original = model.read_text()
model.write_text(original + "\n-- pretend refactor\n")

try:
    res = run_dbt(["ls", "--select", "state:modified+", "--state", "target_base"])
    print("\nnodes selected by state:modified+ :")
    for name in (res.result or []):
        print("  ", name)
finally:
    model.write_text(original)                                  # restore
    shutil.rmtree("target_base", ignore_errors=True)

One edited model selected itself **plus everything downstream** (`+`). In
CI this becomes `dbt build --select state:modified+ --defer --state <prod
manifest>`: build only what a PR touched, borrow everything else from prod.

## 6. This notebook is in the DAG

`_marts__exposures.yml` declares these notebooks as an **exposure**
depending on `fct_orders` + `agg_daily_revenue` -- so `dbt ls --select
+exposure:revenue_notebook` answers "what must be healthy for this notebook
to be trustworthy?". That closes the loop: the warehouse knows about its
consumers.

In [ ]:
res = run_dbt(["ls", "--select", "+exposure:revenue_notebook", "--resource-type", "model"])
for name in (res.result or []):
    print(name)